In [12]:
import torch
import torchvision
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

In [13]:
train_data=torchvision.datasets.CIFAR10(root="./dataset",train=True,transform=torchvision.transforms.ToTensor(),download=True)
val_data=torchvision.datasets.CIFAR10(root="./dataset",train=False,transform=torchvision.transforms.ToTensor(),download=True)
train_data_size=len(train_data)
val_data_len=len(val_data)
train_data_loader=DataLoader(dataset=train_data,batch_size=64,shuffle=True)
val_data_loader=DataLoader(dataset=val_data,batch_size=64,shuffle=True)

Files already downloaded and verified
Files already downloaded and verified


In [ ]:
class My_Net(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.model=nn.Sequential(
            nn.Conv2d(in_channels=3,out_channels=32,kernel_size=5,padding="same"),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(in_channels=32,out_channels=32,kernel_size=5,padding="same"),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(in_channels=32,out_channels=64,kernel_size=5,padding="same"),
            nn.MaxPool2d(kernel_size=2),
            nn.Flatten(),
            nn.Linear(in_features=64*4*4,out_features=64),
            nn.Linear(in_features=64,out_features=10)
        )
    def forward (self,input):
        return self.model(input)
net=My_Net()
loss_F=nn.CrossEntropyLoss()
optimizer=torch.optim.SGD(params=net.parameters(),lr=0.01)
writer=SummaryWriter("./test_log")
train_step=0
epoch=10
for i in range(epoch):
    train_loss=0
    for data in train_data_loader:
        imgs,target = data
        output = net(imgs)
        loss = loss_F(output,target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss+=loss
        train_step+=1
        writer.add_scalar(tag="train_loss",scalar_value=loss,global_step=train_step)
     
    writer.add_scalar(tag="train_totall_loss",scalar_value=train_loss,global_step=i)
    val_loss=0
    with torch.no_grad():
        for data_val in val_data_loader:
            imgs,target = data
            output = net(imgs)
            loss = loss_F(output,target)
            val_loss+=loss        
        writer.add_scalar(tag="val_loss",scalar_value=val_loss,global_step=i)
writer.close()
    

In [15]:
torch.save(net,"test.pth")


使用gpu时需要对模型，损失函数，数据进行cuda操作

In [ ]:
if torch.cuda.is_available():
    net=net.cuda()
    loss_F=loss_F.cuda()
    imgs=imgs.cuda()
    target=target.cuda()

或者device操作

In [16]:
device=torch.device("cpu")
net.to(device)

My_Net(
  (model): Sequential(
    (0): Conv2d(3, 32, kernel_size=(5, 5), stride=(1, 1), padding=same)
    (1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (2): Conv2d(32, 32, kernel_size=(5, 5), stride=(1, 1), padding=same)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1), padding=same)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=1024, out_features=64, bias=True)
    (8): Linear(in_features=64, out_features=10, bias=True)
  )
)